<a href="https://colab.research.google.com/github/AguilarJrr/THE_CORE/blob/main/AAguilarG_CIFAR10.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

## Cargar y Preprocesar Datos CIFAR-10

### Subtask:
Cargar el conjunto de datos CIFAR-10 utilizando `keras.datasets.cifar10` y normalizar los valores de píxeles de las imágenes (por ejemplo, a un rango de 0 a 1).


**Reasoning**:
First, I will import the necessary `cifar10` module from `keras.datasets`. Then, I will load the CIFAR-10 dataset, which includes training and testing images and labels. Finally, I will normalize the pixel values of both the training and testing images by dividing them by 255.0 to scale them to a range of 0 to 1.



In [1]:
import tensorflow as tf
from tensorflow.keras.datasets import cifar10
from tensorflow.keras import Sequential

# 1. Cargar el conjunto de datos CIFAR-10
(x_train, y_train), (x_test, y_test) = cifar10.load_data()

# 2. Normalizar los valores de píxeles de las imágenes
x_train = x_train.astype('float32') / 255.0
x_test = x_test.astype('float32') / 255.0

print(f"Dimensiones de x_train después de la normalización: {x_train.shape}")
print(f"Dimensiones de x_test después de la normalización: {x_test.shape}")
print(f"Valores mínimos de x_train: {x_train.min()}, valores máximos: {x_train.max()}")
print(f"Valores mínimos de x_test: {x_test.min()}, valores máximos: {x_test.max()}")

170498071/170498071 ━━━━━━━━━━━━━━━━━━━━ 4s 0us/step
Dimensiones de x_train después de la normalización: (50000, 32, 32, 3)
Dimensiones de x_test después de la normalización: (10000, 32, 32, 3)
Valores mínimos de x_train: 0.0, valores máximos: 1.0
Valores mínimos de x_test: 0.0, valores máximos: 1.0


## Aplicar Aumentación de Datos

### Subtask:
Implementar técnicas de aumento de datos en las imágenes de entrenamiento para mejorar la robustez del modelo y evitar el sobreajuste. Esto podría incluir rotaciones, cambios de ancho/alto, recortes aleatorios, etc.


**Reasoning**:
I will import `ImageDataGenerator` and define an instance with various data augmentation parameters. Then, I will apply these augmentations to the training data using the `.flow()` method to prepare an augmented data generator for model training.



In [2]:
from tensorflow.keras.preprocessing.image import ImageDataGenerator

# 1. Instantiate ImageDataGenerator with augmentation parameters
datagen = ImageDataGenerator(
    rotation_range=15,          # Rotate images by 15 degrees
    width_shift_range=0.1,      # Shift images horizontally by 10% of total width
    height_shift_range=0.1,     # Shift images vertically by 10% of total height
    horizontal_flip=True,       # Flip images horizontally
    zoom_range=0.1,             # Zoom in/out by 10%
    shear_range=0.1,            # Shear transformations
    fill_mode='nearest'         # Fill in new pixels created by transformations
)

# 2. Fit the ImageDataGenerator to the training data
# This step is optional but can be useful for calculating statistics like mean and std deviation
datagen.fit(x_train)

# 3. Create an augmented data generator for the training data
# This generator will yield batches of augmented images and labels
# The batch_size will be specified later during model training
augmented_train_generator = datagen.flow(x_train, y_train, batch_size=32, shuffle=True)

print("ImageDataGenerator configured and applied to x_train.")
print("Augmented training data generator created.")

ImageDataGenerator configured and applied to x_train.
Augmented training data generator created.


## Cargar Modelo Pre-entrenado (Transfer Learning)

### Subtask:
Cargar un modelo CNN pre-entrenado (por ejemplo, MobileNetV2, ResNet50, VGG16) sin la capa superior de clasificación, utilizando los pesos entrenados en un conjunto de datos grande como ImageNet. Este servirá como la base para el 'transfer learning'.


**Reasoning**:
I will import the necessary modules, define the input shape, instantiate the MobileNetV2 model without the top classification layer and with ImageNet weights, and then print its summary to verify the model structure.



In [3]:
from tensorflow.keras.applications import MobileNetV2
from tensorflow.keras.layers import Input, Resizing
from tensorflow.keras import Model

# Define the target image size for MobileNetV2
IMG_SIZE = 224

# 1. Define the input layer for CIFAR-10 images (32x32x3)
original_input = Input(shape=(32, 32, 3))

# 2. Add a resizing layer to scale images to MobileNetV2's expected input size
resized_input = Resizing(IMG_SIZE, IMG_SIZE, interpolation="bilinear")(original_input)

# 3. Instantiate the MobileNetV2 model without the top classification layer (include_top=False)
#    and loading the pre-trained ImageNet weights (weights='imagenet').
#    The input_shape to MobileNetV2 should now match the resized_input dimensions.
base_model = MobileNetV2(input_shape=(IMG_SIZE, IMG_SIZE, 3),
                           include_top=False,
                           weights='imagenet')

# Freeze the base model layers so they are not updated during training
base_model.trainable = False

# 4. Create a new model that includes the resizing and the base_model
#    The functional API is used here to connect the layers.
model_input = original_input
model_output = base_model(resized_input)

# Define a new 'base_model_with_resizing' that takes 32x32x3 input and outputs MobileNetV2 features
base_model_with_resizing = Model(inputs=model_input, outputs=model_output, name="MobileNetV2_Features")

# Print a summary of the new base model with resizing to verify its structure.
base_model_with_resizing.summary()

print("MobileNetV2 base model with input resizing loaded successfully with ImageNet weights and without the top classification layer.")
print("Base model layers have been frozen.")

9406464/9406464 ━━━━━━━━━━━━━━━━━━━━ 0s 0us/step


Model: "MobileNetV2_Features"

┏━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━┓
┃ Layer (type)                    ┃ Output Shape           ┃       Param # ┃
┡━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━┩
│ input_layer (InputLayer)        │ (None, 32, 32, 3)      │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ resizing (Resizing)             │ (None, 224, 224, 3)    │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ mobilenetv2_1.00_224            │ (None, 7, 7, 1280)     │     2,257,984 │
│ (Functional)                    │                        │               │
└─────────────────────────────────┴────────────────────────┴───────────────┘

 Total params: 2,257,984 (8.61 MB)

 Trainable params: 0 (0.00 B)

 Non-trainable params: 2,257,984 (8.61 MB)

MobileNetV2 base model with input resizing loaded successfully with ImageNet weights and without the top classification layer.
Base model layers have been frozen.


**Reasoning**:
The next step is to add custom classification layers on top of the pre-trained and frozen MobileNetV2 base model to adapt it for the CIFAR-10 dataset. This typically involves flattening the output of the base model, adding one or more Dense layers, and potentially Batch Normalization and Dropout layers for improved performance and regularization.



In [4]:
from tensorflow.keras.layers import GlobalAveragePooling2D, Dense, Dropout, BatchNormalization

# 1. Obtener la salida del modelo base (después del resizing y MobileNetV2)
x = base_model_with_resizing.output

# 2. Añadir capas de clasificación personalizadas
# GlobalAveragePooling2D reduce la dimensionalidad espacial, preparando la salida para las capas Dense
x = GlobalAveragePooling2D()(x)

# Capa Dense con BatchNormalization y ReLU para feature extraction
x = Dense(units=256, activation='relu')(x)
x = BatchNormalization()(x)
x = Dropout(0.5)(x)

# Capa Dense de salida para la clasificación de 10 clases (CIFAR-10)
# 'softmax' para clasificación multiclase
NUM_CLASSES = 10 # CIFAR-10 tiene 10 clases
predictions = Dense(units=NUM_CLASSES, activation='softmax')(x)

# 3. Crear el modelo completo
model = Model(inputs=base_model_with_resizing.input, outputs=predictions)

# 4. Imprimir un resumen del modelo para verificar la estructura y los parámetros
model.summary()

print("Custom classification layers added and full model created.")
print("The base model is frozen, and only the new layers are trainable by default.")


Model: "functional"

┏━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━┓
┃ Layer (type)                    ┃ Output Shape           ┃       Param # ┃
┡━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━┩
│ input_layer (InputLayer)        │ (None, 32, 32, 3)      │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ resizing (Resizing)             │ (None, 224, 224, 3)    │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ mobilenetv2_1.00_224            │ (None, 7, 7, 1280)     │     2,257,984 │
│ (Functional)                    │                        │               │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ global_average_pooling2d        │ (None, 1280)           │             0 │
│ (GlobalAveragePooling2D)        │                        │               │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense (Dense)                   │ (None, 256)            │       327,936 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ batch_normalization             │ (None, 256)            │         1,024 │
│ (BatchNormalization)            │                        │               │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dropout (Dropout)               │ (None, 256)            │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense_1 (Dense)                 │ (None, 10)             │         2,570 │
└─────────────────────────────────┴────────────────────────┴───────────────┘

 Total params: 2,589,514 (9.88 MB)

 Trainable params: 331,018 (1.26 MB)

 Non-trainable params: 2,258,496 (8.62 MB)

Custom classification layers added and full model created.
The base model is frozen, and only the new layers are trainable by default.


# Task
Compilar el Modelo: Configurar el modelo para el entrenamiento con un optimizador adecuado (por ejemplo, Adam), una función de pérdida ('sparse_categorical_crossentropy') y métricas ('accuracy').

## Compilar el Modelo

### Subtask:
Configurar el modelo para el entrenamiento con un optimizador adecuado (por ejemplo, Adam), una función de pérdida ('sparse_categorical_crossentropy') y métricas ('accuracy').


**Reasoning**:
I will compile the model using the specified optimizer, loss function, and metrics. This prepares the model for training.



In [5]:
import tensorflow as tf

# 1. Compile the model
# 2. Specify the Adam optimizer
# 3. Define 'sparse_categorical_crossentropy' as the loss function
# 4. Include 'accuracy' as the metric
model.compile(optimizer=tf.keras.optimizers.Adam(),
              loss='sparse_categorical_crossentropy',
              metrics=['accuracy'])

print("Model compiled successfully with Adam optimizer, sparse_categorical_crossentropy loss, and accuracy metric.")

Model compiled successfully with Adam optimizer, sparse_categorical_crossentropy loss, and accuracy metric.


# Task
**Subtask:**
Entrenar el modelo utilizando los datos aumentados (`augmented_train_generator`) y los datos de validación (`x_test`, `y_test`), registrando el historial de entrenamiento.

**Reasoning:**
I will train the model using the `fit` method. For training data, I will use `augmented_train_generator` to leverage the data augmentation techniques. For validation, I will use the `x_test` and `y_test` datasets. I will specify a suitable number of epochs and steps per epoch, and store the returned `history` object to monitor training progress.

## Entrenar el Modelo

### Subtask:
Entrenar el modelo utilizando los datos aumentados (`augmented_train_generator`) y los datos de validación (`x_test`, `y_test`), registrando el historial de entrenamiento.


**Reasoning**:
I will train the compiled model using the augmented training data and validation data, storing the training history as instructed.



In [6]:
import math

# 1. Determinar el número de pasos por época
batch_size = 32 # El mismo batch_size usado en ImageDataGenerator
steps_per_epoch = math.ceil(len(x_train) / batch_size)

# 2. Entrenar el modelo
history_fine_tuning = model.fit(
    augmented_train_generator,
    steps_per_epoch=steps_per_epoch,
    epochs=10, # Establecer un número adecuado de épocas
    validation_data=(x_test, y_test)
)

print("Model training completed.")

Epoch 1/10
1563/1563 ━━━━━━━━━━━━━━━━━━━━ 98s 50ms/step - accuracy: 0.6557 - loss: 1.0064 - val_accuracy: 0.7770 - val_loss: 0.6475
Epoch 2/10
1563/1563 ━━━━━━━━━━━━━━━━━━━━ 54s 34ms/step - accuracy: 0.7017 - loss: 0.8628 - val_accuracy: 0.7692 - val_loss: 0.6671
Epoch 3/10
1563/1563 ━━━━━━━━━━━━━━━━━━━━ 83s 35ms/step - accuracy: 0.7070 - loss: 0.8426 - val_accuracy: 0.7737 - val_loss: 0.6536
Epoch 4/10
1563/1563 ━━━━━━━━━━━━━━━━━━━━ 82s 35ms/step - accuracy: 0.7150 - loss: 0.8295 - val_accuracy: 0.7799 - val_loss: 0.6417
Epoch 5/10
1563/1563 ━━━━━━━━━━━━━━━━━━━━ 55s 35ms/step - accuracy: 0.7160 - loss: 0.8243 - val_accuracy: 0.7900 - val_loss: 0.6007
Epoch 6/10
1563/1563 ━━━━━━━━━━━━━━━━━━━━ 82s 36ms/step - accuracy: 0.7212 - loss: 0.8096 - val_accuracy: 0.7948 - val_loss: 0.5941
Epoch 7/10
1563/1563 ━━━━━━━━━━━━━━━━━━━━ 56s 36ms/step - accuracy: 0.7287 - loss: 0.7966 - val_accuracy: 0.7794 - val_loss: 0.6352
Epoch 8/10
1563/1563 ━━━━━━━━━━━━━━━━━━━━ 82s 35ms/step - accuracy: 0.7263 -

# Task
Descongelar algunas capas del modelo base (`base_model`) para permitir su entrenamiento durante la etapa de fine-tuning, manteniendo una porción inicial del modelo congelada para preservar las características de bajo nivel.

## Preparar para Fine-tuning

### Subtask:
Descongelar algunas capas del modelo base (MobileNetV2) para permitir su entrenamiento durante el fine-tuning, manteniendo una porción inicial congelada.


**Reasoning**:
To perform fine-tuning, the pre-trained `base_model` (MobileNetV2) needs to be unfrozen to allow its weights to be updated during training. However, it's common practice to keep the initial layers frozen, as they often capture general low-level features (edges, textures) that are useful across many tasks and may be sensitive to changes. Therefore, I will first unfreeze the entire `base_model` and then iterate through its layers to selectively freeze the initial layers. Finally, I will print the model summary to confirm that the desired layers are set to trainable.

**Reasoning**:
I will implement the fine-tuning preparation by first unfreezing the `base_model` (MobileNetV2), then re-freezing a specified number of initial layers to preserve low-level features, and finally print the model summary to confirm the changes in trainable parameters.



In [8]:
import tensorflow as tf

# 1. Acceder al modelo base de MobileNetV2 (que está dentro de base_model_with_resizing)
# 'base_model_with_resizing' tiene una capa MobileNetV2 como su única capa significativa más allá de la entrada.
# Accedemos a la instancia de MobileNetV2 directamente.

# Assuming `base_model` is the actual MobileNetV2 instance from the previous cell
# If it's not directly accessible, we might need to retrieve it from `base_model_with_resizing.layers`
# In our case, `base_model` is already defined as the MobileNetV2 instance

# 2. Descongelar todo el modelo base de MobileNetV2
base_model.trainable = True

# 3. Recorrer las capas del modelo base de MobileNetV2 y congelar las primeras 'N' capas
# Es una buena práctica congelar las capas iniciales que aprenden características genéricas.
# MobileNetV2 tiene aproximadamente 155 capas en su modelo base.
# Elegiremos congelar las primeras 100 capas, dejando ~55 capas para entrenar.

FROZEN_LAYERS_COUNT = 100 # Puedes ajustar este número

for i, layer in enumerate(base_model.layers):
    if i < FROZEN_LAYERS_COUNT:
        layer.trainable = False
    else:
        layer.trainable = True

# 4. Recompilar el modelo completo después de cambiar el estado de entrenamiento de las capas del modelo base
# Esto es crucial para que los cambios en 'trainable' de las capas tengan efecto.
model.compile(optimizer=tf.keras.optimizers.Adam(learning_rate=1e-5), # Usar un learning rate más bajo para fine-tuning
              loss='sparse_categorical_crossentropy',
              metrics=['accuracy'])

# 5. Imprimir un resumen del modelo completo para verificar los parámetros entrenables
model.summary()

print(f"MobileNetV2 base model unfrozen. The first {FROZEN_LAYERS_COUNT} layers are now frozen, and the rest are trainable for fine-tuning.")
print("Model recompiled with a lower learning rate for fine-tuning.")

Model: "functional"

┏━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━┓
┃ Layer (type)                    ┃ Output Shape           ┃       Param # ┃
┡━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━┩
│ input_layer (InputLayer)        │ (None, 32, 32, 3)      │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ resizing (Resizing)             │ (None, 224, 224, 3)    │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ mobilenetv2_1.00_224            │ (None, 7, 7, 1280)     │     2,257,984 │
│ (Functional)                    │                        │               │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ global_average_pooling2d        │ (None, 1280)           │             0 │
│ (GlobalAveragePooling2D)        │                        │               │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense (Dense)                   │ (None, 256)            │       327,936 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ batch_normalization             │ (None, 256)            │         1,024 │
│ (BatchNormalization)            │                        │               │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dropout (Dropout)               │ (None, 256)            │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense_1 (Dense)                 │ (None, 10)             │         2,570 │
└─────────────────────────────────┴────────────────────────┴───────────────┘

 Total params: 2,589,514 (9.88 MB)

 Trainable params: 2,192,458 (8.36 MB)

 Non-trainable params: 397,056 (1.51 MB)

MobileNetV2 base model unfrozen. The first 100 layers are now frozen, and the rest are trainable for fine-tuning.
Model recompiled with a lower learning rate for fine-tuning.


# Task
## Reentrenar (Fine-tuning) el Modelo

### Subtask:
Entrenar el modelo con las capas del modelo base parcialmente descongeladas y el learning rate ajustado para fine-tuning, utilizando los datos aumentados (`augmented_train_generator`) y los datos de validación (`x_test`, `y_test`), registrando el historial de entrenamiento para observar las mejoras.

### Reasoning:
After recompiling the model with a lower learning rate and unfreezing some layers of the pre-trained `base_model` for fine-tuning, the next step is to re-train the model. This fine-tuning phase will allow the model to learn more specific features relevant to the CIFAR-10 dataset. I will use the `fit` method with `augmented_train_generator` for robust training and `x_test`, `y_test` for validation, storing the training `history` to analyze the performance improvements. I will consider increasing the number of epochs slightly compared to the initial training phase to allow for sufficient fine-tuning.

## Recompilar y Reentrenar (Fine-tuning)

### Subtask:
Entrenar el modelo con las capas del modelo base parcialmente descongeladas y el learning rate ajustado para fine-tuning, utilizando los datos aumentados (augmented_train_generator) y los datos de validación (x_test, y_test), registrando el historial de entrenamiento para observar las mejoras.


## Evaluar el Modelo

### Subtask:
Evaluar el rendimiento final del modelo en el conjunto de datos de prueba (x_test, y_test) y mostrar las métricas clave como la precisión y la pérdida.


**Reasoning**:
To evaluate the model's final performance, I will use the `evaluate()` method on the test dataset (`x_test`, `y_test`) and print the resulting loss and accuracy.



In [9]:
import numpy as np

# 1. Evaluar el modelo en el conjunto de datos de prueba
loss, accuracy = model.evaluate(x_test, y_test)

# 2. Almacenar los resultados de la evaluación (pérdida y precisión) en variables
# Already done in the previous step by assigning to 'loss' and 'accuracy'

# 3. Imprimir la pérdida de prueba y la precisión de prueba
print(f"Test Loss: {loss:.4f}")
print(f"Test Accuracy: {accuracy:.4f}")

313/313 ━━━━━━━━━━━━━━━━━━━━ 16s 35ms/step - accuracy: 0.7964 - loss: 0.5875
Test Loss: 0.5875
Test Accuracy: 0.7964


# Task
Reentrenar (Fine-tuning) el modelo con las capas del modelo base parcialmente descongeladas y el learning rate ajustado para fine-tuning, utilizando los datos aumentados (`augmented_train_generator`) y los datos de validación (`x_test`, `y_test`), registrando el historial de entrenamiento para observar las mejoras.

## Reentrenar (Fine-tuning) el Modelo

### Subtask:
Entrenar el modelo con las capas del modelo base parcialmente descongeladas y el learning rate ajustado para fine-tuning, utilizando los datos aumentados (augmented_train_generator) y los datos de validación (x_test, y_test), registrando el historial de entrenamiento para observar las mejoras.


**Reasoning**:
I will train the model for fine-tuning using the `model.fit()` method with the augmented training data and the test set for validation, storing the training history to observe the improvements.



In [ ]:
import math

# 1. Determinar el número de pasos por época
batch_size = 32 # El mismo batch_size usado en ImageDataGenerator
steps_per_epoch = math.ceil(len(x_train) / batch_size)

# 2. Entrenar el modelo con las capas base parcialmente descongeladas y el learning rate ajustado
history_fine_tuning_phase = model.fit(
    augmented_train_generator,
    steps_per_epoch=steps_per_epoch,
    epochs=20, # Aumentar el número de épocas para fine-tuning
    validation_data=(x_test, y_test)
)

print("Model fine-tuning completed.")

Epoch 1/20
1563/1563 ━━━━━━━━━━━━━━━━━━━━ 109s 55ms/step - accuracy: 0.6707 - loss: 0.9836 - val_accuracy: 0.8172 - val_loss: 0.5494
Epoch 2/20
1130/1563 ━━━━━━━━━━━━━━━━━━━━ 18s 43ms/step - accuracy: 0.7381 - loss: 0.7732

**Reasoning**:
I will train the model for fine-tuning using the `model.fit()` method with the augmented training data and the test set for validation, storing the training history to observe the improvements.



In [ ]:
import math
import tensorflow as tf
from tensorflow.keras.datasets import cifar10
from tensorflow.keras.preprocessing.image import ImageDataGenerator
from tensorflow.keras.applications import MobileNetV2
from tensorflow.keras.layers import Input, Resizing, GlobalAveragePooling2D, Dense, Dropout, BatchNormalization
from tensorflow.keras import Model

# Re-load CIFAR-10 data and normalize pixels to ensure x_train and x_test are defined
(x_train, y_train), (x_test, y_test) = cifar10.load_data()
x_train = x_train.astype('float32') / 255.0
x_test = x_test.astype('float32') / 255.0

# Re-define ImageDataGenerator and augmented_train_generator
datagen = ImageDataGenerator(
    rotation_range=15,          # Rotate images by 15 degrees
    width_shift_range=0.1,      # Shift images horizontally by 10% of total width
    height_shift_range=0.1,     # Shift images vertically by 10% of total height
    horizontal_flip=True,       # Flip images horizontally
    zoom_range=0.1,             # Zoom in/out by 10%
    shear_range=0.1,            # Shear transformations
    fill_mode='nearest'         # Fill in new pixels created by transformations
)
datagen.fit(x_train)
augmented_train_generator = datagen.flow(x_train, y_train, batch_size=32, shuffle=True)


# Re-define the base model with resizing
IMG_SIZE = 224
original_input = Input(shape=(32, 32, 3))
resized_input = Resizing(IMG_SIZE, IMG_SIZE, interpolation="bilinear")(original_input)
base_model = MobileNetV2(input_shape=(IMG_SIZE, IMG_SIZE, 3),
                           include_top=False,
                           weights='imagenet')
# Create a new model that includes the resizing and the base_model
model_input = original_input
model_output = base_model(resized_input)
base_model_with_resizing = Model(inputs=model_input, outputs=model_output, name="MobileNetV2_Features")


# Re-add custom classification layers and define the full model
x = base_model_with_resizing.output
x = GlobalAveragePooling2D()(x)
x = Dense(units=256, activation='relu')(x)
x = BatchNormalization()(x)
x = Dropout(0.5)(x)
NUM_CLASSES = 10
predictions = Dense(units=NUM_CLASSES, activation='softmax')(x)
model = Model(inputs=base_model_with_resizing.input, outputs=predictions)

# Set up for fine-tuning: unfreeze some base model layers and recompile
base_model.trainable = True # First unfreeze all
FROZEN_LAYERS_COUNT = 100
for i, layer in enumerate(base_model.layers):
    if i < FROZEN_LAYERS_COUNT:
        layer.trainable = False
    else:
        layer.trainable = True

# Recompile the model with a lower learning rate for fine-tuning
model.compile(optimizer=tf.keras.optimizers.Adam(learning_rate=1e-5),
              loss='sparse_categorical_crossentropy',
              metrics=['accuracy'])


# 1. Determinar el número de pasos por época
batch_size = 32 # El mismo batch_size usado en ImageDataGenerator
steps_per_epoch = math.ceil(len(x_train) / batch_size)

# 2. Entrenar el modelo con las capas base parcialmente descongeladas y el learning rate ajustado
history_fine_tuning_phase = model.fit(
    augmented_train_generator,
    steps_per_epoch=steps_per_epoch,
    epochs=20, # Aumentar el número de épocas para fine-tuning
    validation_data=(x_test, y_test)
)

print("Model fine-tuning completed.")